In [1]:
# %% [code]
import os
import pandas as pd
import ee
import time
import requests

project = "landslide-identification-nepal" #The googel earth engine project name
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv" #location of .csv containing landslide incidents
start_index = 0  #starting range of images to be downloaded
end_index = 1

DOWNLOAD_DIR = '/kaggle/working/downloads'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)   # folder name in your Google Drive

gee_key = "/kaggle/input/datasets/sanjayashrestha123/gee-key/landslide-identification-nepal-cccd90850069.json"
service_account = 'kaggle-import@landslide-identification-nepal.iam.gserviceaccount.com'

credentials = ee.ServiceAccountCredentials(service_account, gee_key)

try:
    ee.Initialize(credentials, project=project)
except Exception as e:
    print("EE initialization failed.")
    raise e

csv_filename = input_csv
df = pd.read_csv(csv_filename)
df = df.iloc[721:]
df['incident_on'] = pd.to_datetime(df['incident_on'])

MAX_AOI_DEG = 0.1


def mask_s2_clouds(image):
    scl = image.select('SCL')
    clean_mask = (scl.eq(2).bitwiseOr(scl.eq(4))
                           .bitwiseOr(scl.eq(5))
                           .bitwiseOr(scl.eq(6))
                           .bitwiseOr(scl.eq(7))
                           .bitwiseOr(scl.eq(11)))
    return image.updateMask(clean_mask)

def clamp_aoi(min_lon, min_lat, max_lon, max_lat):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat

    if lon_span <= MAX_AOI_DEG and lat_span <= MAX_AOI_DEG:
        # Area is within limit — return as-is
        return min_lon, min_lat, max_lon, max_lat

    # Area exceeds limit — clamp to MAX_AOI_DEG centered on the bbox center
    cx = (min_lon + max_lon) / 2
    cy = (min_lat + max_lat) / 2
    half = MAX_AOI_DEG / 2
    return cx - half, cy - half, cx + half, cy + half

def download_image(image, aoi, filename, scale=10):
    try:
        url = image.getDownloadURL({
            'scale': scale,
            'region': aoi,
            'format': 'GeoTIFF',
            'crs': 'EPSG:4326',
        })
        response = requests.get(url, stream=True, timeout=300)
        response.raise_for_status()
        filepath = f'{DOWNLOAD_DIR}/{filename}.tif'
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"   ✅ Downloaded: {filepath}")
    except Exception as e:
        print(f"   ❌ Failed to download {filename}: {e}")

def submit_landslide_export(incident_id, search_window_days=120):
    row = df[df['id'] == incident_id]
    if row.empty:
        print(f"❌ ID {incident_id} not found.")
        return

    row = row.iloc[0]
    incident_date = row['incident_on']
    c_min_lon, c_min_lat, c_max_lon, c_max_lat = clamp_aoi(
        row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat']
    )
    aoi = ee.Geometry.Rectangle([c_min_lon, c_min_lat, c_max_lon, c_max_lat])

    before_target = incident_date - pd.DateOffset(months=18)
    after_target  = incident_date + pd.DateOffset(months=18)
    half = pd.DateOffset(days=search_window_days // 2)

    before_start = (before_target - half).strftime('%Y-%m-%d')
    before_end   = (before_target + half).strftime('%Y-%m-%d')
    after_start  = (after_target  - half).strftime('%Y-%m-%d')
    after_end    = (after_target  + half).strftime('%Y-%m-%d')

    print(f"\n📋 Checking ID {incident_id}: {row['title']}")

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
            .map(mask_s2_clouds))

    bands = ['B4', 'B3', 'B2', 'B8']

    periods = [
        ('before', s2.filterDate(before_start, before_end)),
        ('after',  s2.filterDate(after_start,  after_end)),
    ]

    # Validate first
    validated = {}
    for label, collection in periods:
        count = collection.size().getInfo()
        if count == 0:
            print(f"⏭️  Skipping ID {incident_id} — {label} window has no scenes.")
            return
        print(f"✅ {label}: {count} scene(s) found.")
        validated[label] = collection

    print(f"🚀 Downloading images for ID {incident_id}...")

    # Download before/after
    for label, collection in validated.items():
        img = collection.median().select(bands).clip(aoi)
        download_image(img, aoi, f'incident_{incident_id}_{label}', scale=10)

    # Download slope
    dem = ee.Image('USGS/SRTMGL1_003')
    slope = ee.Terrain.slope(dem).clip(aoi)
    download_image(slope, aoi, f'incident_{incident_id}_slope', scale=30)

def monitor_tasks(tasks, poll_interval=30):
    """Poll submitted tasks until all complete or fail."""
    print(f"\n⏳ Monitoring {len(tasks)} tasks (checking every {poll_interval}s)...")
    pending = list(tasks)

    while pending:
        still_pending = []
        for name, task in pending:
            status = task.status()['state']
            if status == 'COMPLETED':
                print(f"✅ {name}: done")
            elif status == 'FAILED':
                print(f"❌ {name}: FAILED — {task.status().get('error_message', '')}")
            else:
                still_pending.append((name, task))  # READY or RUNNING

        pending = still_pending
        if pending:
            print(f"   {len(pending)} still running...")
            time.sleep(poll_interval)

    print("🎉 All tasks finished.")

# --- Submit all tasks ---
all_tasks = []

# Single incident
# tasks = submit_landslide_export(47070)
# all_tasks.extend(tasks)

# Or batch — first 10
for inc_id in df['id'].iloc[start_index:end_index]:
    submit_landslide_export(inc_id)

# --- Optional: wait and monitor ---
monitor_tasks(all_tasks)

/tmp/ipykernel_23/679650575.py:30: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['incident_on'] = pd.to_datetime(df['incident_on'])



📋 Checking ID 75849: Landslide at Palata Rural Municipality-7
✅ before: 10 scene(s) found.
✅ after: 7 scene(s) found.
🚀 Downloading images for ID 75849...
   ✅ Downloaded: /kaggle/working/downloads/incident_75849_before.tif
   ✅ Downloaded: /kaggle/working/downloads/incident_75849_after.tif
   ✅ Downloaded: /kaggle/working/downloads/incident_75849_slope.tif

⏳ Monitoring 0 tasks (checking every 30s)...
🎉 All tasks finished.
